# M04 lab solution: prompt bake-off

**Brief.** Write three prompts for grading whether a retrieved passage answers a question, and measure their agreement with your own labels on 30 pairs.

`../data/grading_pairs.jsonl` holds 30 pairs from the Larkspur handbook: for 15 golden questions, the top TF-IDF passage that holds an evidence span and the top one that does not. Each label (`yes` or `no`) was written by reading the pair. The three prompts are zero-shot, few-shot and quote-then-decide; each is scored with accuracy and Cohen's kappa from `sklearn.metrics`.

Key: `EXPERIENTIALLABS_API_KEY`.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../../../../src").resolve()))
load_dotenv()

DATA = Path("../data")

import re

import pandas as pd
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix

from helpers.utils import get_experientiallabs_llm
from lab_kit import load_jsonl

In [ ]:
pairs = load_jsonl(DATA / "grading_pairs.jsonl")
labels = [p["label"] for p in pairs]
print(len(pairs), "pairs:", labels.count("yes"), "yes,", labels.count("no"), "no")

llm = get_experientiallabs_llm()  # temperature 0

## 1. Three grading prompts

The few-shot examples come from handbook passages that are not among the 30 pairs, one per label, so the prompt does not see the test set.

In [ ]:
ZERO_SHOT = """Question: {question}

Passage: {passage}

Does the passage answer the question? Reply with one word: yes or no."""

FEW_SHOT = """You decide whether a retrieved passage answers a customer's question.
Answer "yes" only if the passage states the fact the question asks for. A passage on the same topic
that does not state that fact is "no".

Question: How is my data encrypted?
Passage: All data is encrypted at rest with AES-256 and in transit with TLS 1.2 or later. Backups and export archives are encrypted the same way.
Label: yes

Question: How many API requests per minute can a Business key make?
Passage: API keys belong to a project, not to a person. A key can be limited to read-only access, and keys can be rotated from Settings, then API keys, without downtime.
Label: no

Question: {question}
Passage: {passage}
Label:"""

QUOTE_THEN_DECIDE = """Question: {question}

Passage: {passage}

Step 1: Copy the sentence from the passage that answers the question, word for word.
If no sentence answers it, write NONE.
Step 2: On the last line write "Label: yes" if you copied a sentence, or "Label: no" if you wrote NONE."""

PROMPTS = {"zero-shot": ZERO_SHOT, "few-shot": FEW_SHOT, "quote-then-decide": QUOTE_THEN_DECIDE}

## 2. Parse the label

The parser takes the last `yes` or `no` in the reply. A reply with neither is counted as a parse failure and scored as `no`, so a prompt is not rewarded for unreadable output.

In [ ]:
def parse_label(reply: str) -> str | None:
    found = re.findall(r"\b(yes|no)\b", reply.lower())
    return found[-1] if found else None

def grade(prompt: str) -> tuple[list[str], int]:
    preds, failures = [], 0
    for p in pairs:
        reply = llm.invoke(prompt.format(question=p["question"], passage=p["passage"])).content
        label = parse_label(reply)
        failures += label is None
        preds.append(label or "no")
    return preds, failures

## 3. Run the three prompts

In [ ]:
predictions, rows = {}, []
for name, prompt in PROMPTS.items():
    preds, failures = grade(prompt)
    predictions[name] = preds
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=["no", "yes"]).ravel()
    rows.append({"prompt": name, "accuracy": accuracy_score(labels, preds),
                 "cohen_kappa": cohen_kappa_score(labels, preds), "false_yes": fp, "false_no": fn,
                 "parse_failures": failures})
    print(f"{name:<18} done")

## Results

In [ ]:
pd.DataFrame(rows)

In [ ]:
# Pairs where at least one prompt disagrees with the label: read these before changing a prompt.
disagree = pd.DataFrame({"id": [p["id"] for p in pairs], "label": labels, **predictions})
disagree[disagree[list(PROMPTS)].ne(disagree["label"], axis=0).any(axis=1)]

## What this shows

- Kappa corrects accuracy for chance agreement. With 15 `yes` and 15 `no`, a prompt that always says `yes` scores 0.5 accuracy and 0 kappa.
- The negatives are hard on purpose: each is the best-scoring TF-IDF passage on the same topic that lacks the fact. False `yes` on these is the failure that lets a relevance grader pass a near miss.
- Thirty pairs give a coarse ruler. Read the disagreement table before declaring a winner, and add new cases to the labelled set rather than tuning a prompt to one pair.